# Exercice 3 : Les preuves techniques

> **En une phrase :** une preuve, ce n'est pas « oui, c'est configuré » : c'est **la ligne du fichier de configuration** qui le montre.

**Les demandes de l'auditeur traitées ici :** D-03 (configuration sécurisée) et D-04 (sauvegardes, **critique**). Références : CIS Controls v8 (4, 5, 8, 11), ISO 27001 A.8.9, A.8.13, A.8.15.

**Le scénario.** L'auditeur a choisi deux serveurs : `web01` (le serveur Web) et `bd01` (la base de données clients).
L'équipe a extrait leurs fichiers de configuration et leurs journaux dans `donnees/serveurs/`. On les vérifie contrôle par contrôle,
en gardant pour chacun **la ligne qui prouve** le résultat.

**Le plan.**
1. Le piège des lignes commentées.
2. Cinq contrôles de configuration (D-03).
3. À vous : les sauvegardes (D-04).
4. Déposer la pièce et répondre à l'auditeur.

## Étape 0 : exécutez cette cellule en premier
En local, ouvrez **le dossier du chapitre** dans VS Code. Sur Colab, la cellule télécharge le dossier. Elle doit afficher **Prêt : True**.

In [1]:
# Étape 0 : sur Colab, récupère le dossier du chapitre depuis GitHub. En local (VS Code), cette cellule ne fait rien.
import os
if not os.path.exists('auditeur.py'):
    !test -d /content/cours || git clone -q --depth 1 https://github.com/elyesmanai/Code-8INF857---2026.git /content/cours
    %cd "/content/cours/Chapitre 8 - Cadres réglementaires"
print('Prêt :', os.path.exists('auditeur.py'))

Prêt : True


## Étape 1 : Lire un fichier de configuration… correctement
**Ce qu'on fait :** on cherche `PermitRootLogin` (connexion SSH directe en `root`) dans le `sshd_config` de chaque serveur.

In [2]:
from helper import lire_serveur, directive, deposer
import pandas as pd

for serveur in ['web01', 'bd01']:
    print(f'--- {serveur}')
    for ligne in lire_serveur(serveur, 'etc/ssh/sshd_config').splitlines():
        if 'PermitRootLogin' in ligne or 'PasswordAuthentication' in ligne:
            print('   ', ligne)

--- web01
    PermitRootLogin no
    PasswordAuthentication no
--- bd01
    PermitRootLogin yes
    PasswordAuthentication yes


**👀 Ce qu'on doit voir :** des lignes **actives** (`PermitRootLogin no`) et, plus haut dans le fichier, des commentaires (`# To disable tunneled clear text passwords…`).
Une ligne qui commence par `#` ne fait **rien**. Si une directive est absente ou commentée, le logiciel prend sa **valeur par défaut**.

C'est l'erreur classique d'un audit : lire `#PermitRootLogin no` et conclure « conforme ». La fonction `directive()` de `helper.py` ignore les commentaires et renvoie la valeur active **et** la ligne qui la prouve :

In [3]:
print(directive(lire_serveur('bd01', 'etc/ssh/sshd_config'), 'PermitRootLogin'))
print(directive(lire_serveur('bd01', 'etc/ssh/sshd_config'), 'MaxAuthTries'))

('yes', 'PermitRootLogin yes')
(None, None)


**👀 Ce qu'on doit voir :** `('yes', 'PermitRootLogin yes')` : la valeur et sa preuve. Puis `(None, None)` pour `MaxAuthTries`, qui n'est que commentée : SSH utilise sa valeur par défaut (6 essais).

## Étape 2 : Cinq contrôles de configuration (D-03)

| Contrôle | Règle | Référence |
|---|---|---|
| `ssh_root` | `PermitRootLogin no` | CIS 4.7 (comptes par défaut), ISO A.8.2 |
| `ssh_mot_de_passe` | `PasswordAuthentication no` (clés SSH seulement) | CIS 4.1, ISO A.8.5 |
| `mot_de_passe_14` | `minlen` ≥ 14 dans `pwquality.conf` | CIS 5.2 (14 caractères si pas de MFA), ISO A.5.17 |
| `pare_feu` | pare-feu `ufw` actif | CIS 4.4, ISO A.8.20 |
| `journaux_90_jours` | journaux conservés au moins 90 jours (`logrotate`) | CIS 8.10, ISO A.8.15 |

Chaque contrôle donne une ligne : serveur, contrôle, référence, **résultat**, **preuve**, et une **action** si c'est non conforme. La lettre l'a dit : une non-conformité est acceptable **avec un plan d'action**.

In [4]:
ACTIONS = {   # plan d'action si le contrôle échoue
    'ssh_root': "Mettre PermitRootLogin no et passer par sudo ; avant le 15 décembre (M. Gagnon)",
    'ssh_mot_de_passe': "Mettre PasswordAuthentication no après avoir déployé les clés SSH des administrateurs",
    'mot_de_passe_14': "Mettre minlen = 14 dans pwquality.conf ; changement forcé au prochain mot de passe",
    'pare_feu': "Activer ufw : autoriser seulement 22 depuis l'admin et 5432 depuis web01/web02",
    'journaux_90_jours': "Passer logrotate à daily + rotate 90, et envoyer les journaux vers le serveur central",
}

def ligne(serveur, controle, reference, ok, preuve):
    return {'serveur': serveur, 'controle': controle, 'reference': reference,
            'resultat': 'conforme' if ok else 'non conforme', 'preuve': preuve,
            'action': '' if ok else ACTIONS.get(controle, '')}

def verifier_config(serveur):
    sshd = lire_serveur(serveur, 'etc/ssh/sshd_config')
    r = []
    v, p = directive(sshd, 'PermitRootLogin')
    r.append(ligne(serveur, 'ssh_root', 'CIS 4.7 / ISO A.8.2', v == 'no', p or 'directive absente : défaut prohibit-password'))
    v, p = directive(sshd, 'PasswordAuthentication')
    r.append(ligne(serveur, 'ssh_mot_de_passe', 'CIS 4.1 / ISO A.8.5', v == 'no', p or 'directive absente : défaut yes'))
    v, p = directive(lire_serveur(serveur, 'etc/security/pwquality.conf'), 'minlen')
    r.append(ligne(serveur, 'mot_de_passe_14', 'CIS 5.2 / ISO A.5.17', v is not None and int(v) >= 14, p or 'minlen absent : défaut 8'))
    ufw = lire_serveur(serveur, 'ufw_status.txt').splitlines()[0]
    r.append(ligne(serveur, 'pare_feu', 'CIS 4.4 / ISO A.8.20', ufw == 'Status: active', ufw))
    lr = lire_serveur(serveur, 'etc/logrotate.d/rsyslog')
    frequence = next(m for m in ['daily', 'weekly', 'monthly'] if m in lr)
    rotations = int(directive(lr, 'rotate')[0])
    jours = {'daily': 1, 'weekly': 7, 'monthly': 30}[frequence] * rotations
    r.append(ligne(serveur, 'journaux_90_jours', 'CIS 8.10 / ISO A.8.15', jours >= 90, f'{frequence} + rotate {rotations} = {jours} jours'))
    return r

config = pd.DataFrame(verifier_config('web01') + verifier_config('bd01'))
config[['serveur', 'controle', 'resultat', 'preuve']]

,serveur,controle,resultat,preuve
0,web01,ssh_root,conforme,PermitRootLogin no
1,web01,ssh_mot_de_passe,conforme,PasswordAuthentication no
2,web01,mot_de_passe_14,conforme,minlen = 14
3,web01,pare_feu,conforme,Status: active
4,web01,journaux_90_jours,conforme,daily + rotate 90 = 90 jours
5,bd01,ssh_root,non conforme,PermitRootLogin yes
6,bd01,ssh_mot_de_passe,non conforme,PasswordAuthentication yes
7,bd01,mot_de_passe_14,non conforme,minlen absent : défaut 8
8,bd01,pare_feu,non conforme,Status: inactive
9,bd01,journaux_90_jours,non conforme,weekly + rotate 4 = 28 jours


**👀 Ce qu'on doit voir :** `web01` est conforme aux 5 contrôles. `bd01` échoue aux 5 : `root` accessible en SSH, mots de passe acceptés en SSH, `minlen` absent (défaut : 8 caractères), pare-feu **inactif**, journaux gardés **28 jours** au lieu de 90.

**❓ Question pour la classe :** pourquoi garder la ligne `PermitRootLogin no` comme preuve, plutôt qu'écrire simplement « conforme » ?

*Piste : parce que l'auditeur peut la **revérifier** dans le fichier. « Conforme » est une opinion ; la ligne est un fait. Et si quelqu'un modifie le fichier demain, la preuve d'aujourd'hui garde la trace de ce qui était vrai à la date de l'extraction.*

## À vous : les sauvegardes (D-04, critique)
Les deux serveurs écrivent un journal de sauvegarde (`var/log/sauvegarde.log`, outil **restic**). Regardez la fin de celui de `bd01` :

In [5]:
print('\n'.join(lire_serveur('bd01', 'var/log/sauvegarde.log').splitlines()[-18:-12]))

2026-11-09T02:00:09 bd01 restic[2231]: snapshot 109259de saved | files: 186628 | added: 432 MiB | STATUT=OK
2026-11-10T02:00:00 bd01 restic[2231]: snapshot c721a110 saved | files: 184930 | added: 314 MiB | STATUT=OK
2026-11-11T02:00:01 bd01 restic[2231]: snapshot 8d5df6fa saved | files: 182797 | added: 549 MiB | STATUT=OK
2026-11-12T02:00:02 bd01 restic[2231]: Fatal: unable to save snapshot: write /mnt/restic/data: no space left on device | STATUT=ECHEC
2026-11-13T02:00:03 bd01 restic[2231]: Fatal: unable to save snapshot: write /mnt/restic/data: no space left on device | STATUT=ECHEC
2026-11-14T02:00:04 bd01 restic[2231]: Fatal: unable to save snapshot: write /mnt/restic/data: no space left on device | STATUT=ECHEC


Deux contrôles :
- `sauvegarde_24h` (CIS 11.2) : la dernière ligne `STATUT=OK` date d'au plus **1 jour** avant l'extraction (le **26 novembre 2026**) ;
- `test_restauration_12_mois` (CIS 11.5) : une ligne `TEST_RESTAURATION=OK` dans les **365 derniers jours**. Une sauvegarde jamais restaurée n'est pas une sauvegarde : c'est un espoir.

Complétez `verifier_sauvegardes(serveur)`. Elle doit renvoyer **deux** lignes faites avec `ligne(...)`, avec comme preuve la ligne du journal (ou « aucune … » s'il n'y en a pas). La date est dans les 10 premiers caractères de chaque ligne.

In [6]:
from datetime import date
EXTRACTION = date(2026, 11, 26)
ACTIONS['sauvegarde_24h'] = "Libérer l'espace de /mnt/restic, relancer la sauvegarde, alerte si un jour échoue"
ACTIONS['test_restauration_12_mois'] = "Test de restauration de bd01 avant le 30 novembre, puis tous les trimestres"

def verifier_sauvegardes(serveur):
    journal = lire_serveur(serveur, 'var/log/sauvegarde.log').splitlines()
    # À VOUS : trouvez la dernière ligne STATUT=OK et la dernière ligne TEST_RESTAURATION=OK,
    #          puis renvoyez deux lignes : ligne(serveur, 'sauvegarde_24h', 'CIS 11.2 / ISO A.8.13', ok, preuve)
    #                                     ligne(serveur, 'test_restauration_12_mois', 'CIS 11.5 / ISO A.8.13', ok, preuve)
    return []

sauvegardes = pd.DataFrame(verifier_sauvegardes('web01') + verifier_sauvegardes('bd01'))
sauvegardes

""


*Réponse :*
```python
def verifier_sauvegardes(serveur):
    journal = lire_serveur(serveur, 'var/log/sauvegarde.log').splitlines()
    oks = [l for l in journal if 'STATUT=OK' in l]
    tests = [l for l in journal if 'TEST_RESTAURATION=OK' in l]
    ok24 = bool(oks) and (EXTRACTION - date.fromisoformat(oks[-1][:10])).days <= 1
    ok12 = bool(tests) and (EXTRACTION - date.fromisoformat(tests[-1][:10])).days <= 365
    return [ligne(serveur, 'sauvegarde_24h', 'CIS 11.2 / ISO A.8.13', ok24, oks[-1][:80] if oks else 'aucune sauvegarde réussie'),
            ligne(serveur, 'test_restauration_12_mois', 'CIS 11.5 / ISO A.8.13', ok12, tests[-1][:80] if tests else 'aucun test de restauration en 12 mois')]
```
Résultat attendu : `web01` conforme deux fois ; `bd01` non conforme deux fois. Sa dernière sauvegarde réussie date du **11 novembre** (disque plein depuis), et aucun test de restauration n'a jamais été fait.

## Étape 3 : Déposer la pièce D-03 et D-04
**Ce qu'on fait :** les deux tableaux vont dans le même fichier, `controles.csv`.

In [7]:
controles = pd.concat([config, sauvegardes], ignore_index=True)
deposer('controles.csv', controles)
print(controles['resultat'].value_counts().to_string())

Pièce déposée : livrables/controles.csv
resultat
conforme        5
non conforme    5


## Ce que dit l'auditeur
**Ce qu'on fait :** on relance l'auditeur automatique sur tout le dossier `livrables/`. 

In [8]:
import auditeur
auditeur.auditer();

Cabinet Boréal Audit — revue des pièces déposées

⚠️ D-01 Textes applicables et justification          9.3 / 10
      · nis2 : applicabilité « ? » incorrecte
      · nis2 : justification manquante ou trop courte
⚠️ D-02 Inventaire des actifs rapproché du réseau   14.2 / 15  (critique)
      · 10.0.1.47 : aucun propriétaire
      · 10.0.3.58 : aucun propriétaire
✅ D-03 Configuration sécurisée des serveurs        15.0 / 15
❌ D-04 Sauvegardes et test de restauration          0.0 / 10  (critique)
      · web01 / sauvegarde_24h : absent
      · web01 / test_restauration_12_mois : absent
      · bd01 / sauvegarde_24h : absent
      · bd01 / test_restauration_12_mois : absent
❌ D-05 Revue des droits d'accès                     0.0 / 15  (critique)
      · fichier revue_acces.csv introuvable dans livrables/
❌ D-06 Registre et plan de traitement des risques   0.0 / 15
      · fichier registre_risques.csv introuvable dans livrables/
❌ D-07 Registre des incidents et notifications      0.0 / 15  (

**👀 Ce qu'on doit voir :** D-03 ✅ **15 / 15** : toutes les non-conformités de `bd01` ont une preuve et une action. D-04 reste ❌ tant que `verifier_sauvegardes` renvoie une liste vide ; avec votre tâche, ✅ **10 / 10**.
Score : **38 / 100** sans les tâches, **50 / 100** avec les tâches des ex. 1 à 3.

**❓ Question pour la classe :** `bd01` échoue à **tous** les contrôles. Pourtant, l'auditeur peut accepter D-03. Comment est-ce possible ?

*Piste : l'audit ne vérifie pas qu'on est parfait. Il vérifie qu'on **sait** où on en est, preuves à l'appui, et qu'on a un **plan**. Une non-conformité déclarée avec un plan d'action est mineure. La même non-conformité cachée, découverte par l'auditeur, est majeure : elle détruit la confiance dans tout le reste du dossier.*

**❓ Question pour la classe :** `bd01` contient la base clients, sans sauvegarde depuis le 11 novembre, sans pare-feu, avec `root` accessible en SSH par mot de passe. Qu'est-ce que l'équipe doit faire **cette semaine**, avant même l'audit ?

*Piste : réparer la sauvegarde (c'est le risque R1 de l'ex5 : rançongiciel sans sauvegarde). Un audit révèle des risques réels, pas seulement des cases à cocher.*

## Ce qu'il faut retenir
- Une **preuve** est un fait vérifiable : la ligne du fichier, l'entrée du journal, avec sa date. « C'est fait » n'est pas une preuve.
- Attention aux **lignes commentées** et aux **valeurs par défaut** : ce qui compte, c'est la configuration active.
- Une sauvegarde n'existe que si elle **réussit** chaque jour et qu'on a déjà **testé la restauration**.
- Une **non-conformité** déclarée avec un **plan d'action** est acceptable ; cachée, elle ne l'est pas.

**Exercice suivant :** qui a accès à quoi ? La revue des droits d'accès.